# StyleMe — 04: Public-Dataset Slot Benchmark (T-023)

Scores the **shipped** slot classifier (zero-shot prompts + `probes.json` linear head) at scale
against public clothing datasets, on our 6-slot taxonomy (top/bottom/dress/outerwear/shoes/accessory):

| # | dataset | style | why |
|---|---|---|---|
| 1 | DeepFashion2 val | consumer photos, bbox | flat-lay-ish anchor, biggest slot coverage |
| 2 | Fashion Product Images | e-commerce flat-lays | fills shoes + accessory slots |
| 3 | Fashionpedia (HF `detection-datasets/fashionpedia`) | street photos, per-garment bboxes | worn-garment realism, close to on-device "worn" captures |
| 4 | ModaNet | street photos, polygons | second realism set — **experimental**: public annotations may lack a scriptable image source; cell degrades to a skip |

**Runtime:** T4 GPU (free tier). Inputs: `raw/` from notebook 00 (DF2, Fashion Product),
`artifacts/` from 01/02 (`prompt_embeddings.npy`, `taxonomy.json`, `probes.json`).
Outputs: `artifacts/dataset_eval_report.{json,md}` + cached embeddings in `artifacts/`.

**License note:** all four corpora are non-commercial-licensed. Fine for eval/training research
now (T-023/T-009); re-review before any commercial use. They supplement the 20–30 real labeled
phone photos (T-009), never replace them — no public set contains our bed/floor phone flat-lays.


In [ ]:
!pip install -q open-clip-torch pyyaml pandas pillow scikit-learn huggingface_hub datasets

In [ ]:
import os, io, json, glob, ast, subprocess, urllib, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

ROOT = os.environ.get('STYLEME_ROOT')
if not ROOT:
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_notify=20)
        ROOT = '/content/drive/MyDrive/StyleMe'
    except Exception:
        ROOT = '/content/StyleMe'

RAW = os.path.join(ROOT, 'raw')
ART = os.path.join(ROOT, 'artifacts')
os.makedirs(ART, exist_ok=True)
for d in sorted(os.listdir(RAW)):
    print(' raw/', d)

def sh(cmd):
    print('$', ' '.join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError(f'command failed: {cmd[:2]}')
    return r

SEED = 0
rng = np.random.default_rng(SEED)
SLOTS = ['top', 'bottom', 'dress', 'outerwear', 'shoes', 'accessory']
RESULTS = {}   # dataset -> {'zero-shot': {...}, 'probe': {...}}


In [ ]:
import yaml
tax_path = next(iter(filter(os.path.exists, [
    os.path.join(ROOT, 'taxonomy.yaml'),
    '/content/ml/taxonomy.yaml',
    '/content/taxonomy.yaml',
])), None)
assert tax_path, 'taxonomy.yaml not found — upload it to ROOT'
TAX = yaml.safe_load(open(tax_path))

import torch
import open_clip

model = preprocess = tokenizer = None
for name, pretrained in [('MobileCLIP-S1', 'datacompdr'), ('MobileCLIP2-S0', 'dfndr2b')]:
    try:
        print('loading', name, '...', flush=True)
        model, _, preprocess = open_clip.create_model_and_transforms(name, pretrained=pretrained)
        tokenizer = open_clip.get_tokenizer(name)
        break
    except Exception as e:
        print('  failed:', type(e).__name__, str(e)[:200])
assert model is not None, 'no MobileCLIP checkpoint loaded'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
IMG_SIZE = 256
print('device:', device)

@torch.no_grad()
def embed_images(pils, batch=64):
    out = []
    for i in range(0, len(pils), batch):
        tens = torch.stack([preprocess(p) for p in pils[i:i+batch]]).to(device)
        e = model.encode_image(tens).float()
        e = e / e.norm(dim=-1, keepdim=True)
        out.append(e.cpu().numpy())
    return np.vstack(out).astype(np.float32)

def load_pil(path, bbox=None, size=256):
    from PIL import Image
    try:
        im = Image.open(path).convert('RGB')
    except Exception:
        return None
    if bbox is not None:
        x1, y1, x2, y2 = [float(v) for v in bbox]
        W, H = im.size
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(W, int(x2)), min(H, int(y2))
        if x2 - x1 < 8 or y2 - y1 < 8:
            return None
        im = im.crop((x1, y1, x2, y2))
    return im.resize((size, size))


In [ ]:
# The shipped heads: zero-shot prompt sims + probes.json linear slot head.
prompt_emb = np.load(os.path.join(ART, 'prompt_embeddings.npy'))
meta = json.load(open(os.path.join(ART, 'taxonomy.json')))
prompts = meta['prompts']
slot_idx = [i for i, p in enumerate(prompts) if p.startswith('slot:')]
slot_vals = [prompts[i].split(':', 1)[1] for i in slot_idx]
print('slot prompts:', slot_vals)

probes = json.load(open(os.path.join(ART, 'probes.json')))
slot_probe = probes['slot']
Wp = np.array(slot_probe['W'], dtype=np.float32)   # (6, 512)
bp = np.array(slot_probe['b'], dtype=np.float32)    # (6,)
probe_labels = list(slot_probe['labels'])
print('probe labels:', probe_labels, '| reported acc:', slot_probe['acc'], 'n:', slot_probe['n'])

def pred_zero_shot(emb):
    sims = emb @ prompt_emb[slot_idx].T
    return [slot_vals[int(i)] for i in sims.argmax(1)]

def pred_probe(emb):
    logits = emb @ Wp.T + bp
    return [probe_labels[int(i)] for i in logits.argmax(1)]

def score(y_true, y_pred, n_expected=None):
    from sklearn.metrics import confusion_matrix, classification_report
    present = sorted(set(y_true) | set(y_pred))
    acc = float(np.mean([t == p for t, p in zip(y_true, y_pred)]))
    per_slot = {}
    for s in SLOTS:
        mask = [t == s for t in y_true]
        if any(mask):
            hit = [p == t for p, t in zip(y_pred, y_true) if t == s]
            per_slot[s] = {'recall': float(np.mean(hit)), 'n': int(len(hit))}
    cm = confusion_matrix(y_true, y_pred, labels=present).tolist()
    return {'acc': acc, 'n': len(y_true), 'per_slot': per_slot,
            'confusion_labels': present, 'confusion': cm}

def embed_cached(name, paths_bboxes):
    cache = os.path.join(ART, f'ds04_{name}_emb.npz')
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        return list(z['emb']), list(z['labels'])
    pils, labels = [], []
    for path, bbox, label in paths_bboxes:
        im = load_pil(path, bbox=bbox)
        if im is not None:
            pils.append(im); labels.append(label)
    if not pils:
        return [], []
    emb = embed_images(pils)
    np.savez_compressed(cache, emb=emb, labels=np.array(labels, dtype=object))
    return list(emb), labels


## 1. DeepFashion2 validation — anchor

Full validation split (32k), category→slot via `taxonomy.yaml` `slots.*.categories`
(1,2,5,6→top; 3,4→outerwear; 7,8,9→bottom; 10–13→dress). Categories 14+ (keep warm beyond)
are unmapped in our taxonomy and skipped. `N_PER_SLOT=2000` keeps a T4 pass ≈ minutes.


In [ ]:
NAME = 'deepfashion2'
df2 = pd.read_csv(os.path.join(RAW, 'deepfashion2', 'validation.csv'))
CAT2SLOT = {}
for slot, spec in TAX['slots'].items():
    for cid in spec.get('categories') or []:
        CAT2SLOT[int(cid)] = slot
df2['slot'] = df2['category_id'].map(CAT2SLOT)
df2 = df2.dropna(subset=['slot']).copy()
print(df2['slot'].value_counts().to_string())

N_PER_SLOT = 2000
rows = (df2.groupby('slot', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), N_PER_SLOT), random_state=SEED))
          .reset_index(drop=True))
img_dir = os.path.join(RAW, 'deepfashion2', 'images')
paths = []
for _, row in rows.iterrows():
    p = os.path.join(img_dir, os.path.basename(str(row['path'])))
    bb = row['b_box']
    if isinstance(bb, str):
        bb = ast.literal_eval(bb)
    paths.append((p, bb, row['slot']))
emb, y_true = embed_cached(NAME, paths)
print('crops:', len(emb))

if len(emb) > 0:
    RESULTS[NAME] = {'zero-shot': score(y_true, pred_zero_shot(emb)),
                     'probe': score(y_true, pred_probe(emb))}
    print('probe acc:', RESULTS[NAME]['probe']['acc'])
    print(json.dumps(RESULTS[NAME]['probe']['per_slot'], indent=1))


## 2. Fashion Product Images — shoes + accessory gap

E-commerce flat-lays. Mapping (defensive — unknowns skipped with a warning):
Footwear→shoes; masterCategory Accessories→accessory; Apparel Topwear→top,
Bottomwear→bottom, Dress→dress. Innerwear/Saree/Loungewear and Personal Care etc. skipped.


In [ ]:
NAME = 'fashion_product'
styles_csv = os.path.join(RAW, 'fashion-product', 'styles.csv')
img_dir = os.path.join(RAW, 'fashion-product', 'images')
styles = pd.read_csv(styles_csv, on_bad_lines='skip')

def fpi_slot(row):
    sub = str(row.get('subCategory', '')).strip()
    if sub == 'Topwear': return 'top'
    if sub == 'Bottomwear': return 'bottom'
    if sub == 'Dress': return 'dress'
    if sub == 'Shoes': return 'shoes'
    if str(row.get('masterCategory', '')).strip() == 'Accessories':
        return 'accessory'
    return None

styles['slot'] = styles.apply(fpi_slot, axis=1)
styles = styles.dropna(subset=['slot']).copy()
styles['id'] = styles['id'].astype(int).astype(str)
print(styles['slot'].value_counts().to_string())

N_PER_SLOT = 2000
rows = (styles.groupby('slot', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), N_PER_SLOT), random_state=SEED))
          .reset_index(drop=True))
paths = [(os.path.join(img_dir, r['id'] + '.jpg'), None, r['slot'])
         for _, r in rows.iterrows()]
emb, y_true = embed_cached(NAME, paths)
print('crops:', len(emb))

if len(emb) > 0:
    RESULTS[NAME] = {'zero-shot': score(y_true, pred_zero_shot(emb)),
                     'probe': score(y_true, pred_probe(emb))}
    print('probe acc:', RESULTS[NAME]['probe']['acc'])
    print(json.dumps(RESULTS[NAME]['probe']['per_slot'], indent=1))
else:
    print('no Fashion Product images found — run notebook 00 first (section: fashion-product)')


## 3. Fashionpedia (HF) — street realism

Per-garment bounding boxes on worn outfits — closest public proxy for on-device
"worn" photos (Q-003). Uses HF `detection-datasets/fashionpedia`; images come from
the hub, annotations are per-object with category names.


In [ ]:
NAME = 'fashionpedia'
# Fashionpedia attribute-taxonomy names mapped to our 6 slots; garment parts
# (collar, sleeve, pocket, lapel, epaulette, neckline, bow, hood) are skipped.
FP_LABELS_TO_SLOT = {
    'shirt, blouse': 'top', 'top, t-shirt, sweatshirt': 'top', 'sweater': 'top',
    'cardigan': 'top',
    'jacket': 'outerwear', 'vest': 'outerwear', 'coat': 'outerwear', 'cape': 'outerwear',
    'pants': 'bottom', 'shorts': 'bottom', 'skirt': 'bottom',
    'dress, jumpsuit': 'dress',
    'shoe': 'shoes',
    'bag, wallet': 'accessory', 'belt': 'accessory', 'glasses': 'accessory',
    'hat': 'accessory', 'headband, head covering, hair accessory': 'accessory',
    'tie': 'accessory', 'glove': 'accessory', 'watch': 'accessory',
    'scarf': 'accessory', 'umbrella': 'accessory',
}

try:
    from datasets import load_dataset
    ds = load_dataset('detection-datasets/fashionpedia', split='validation')
    names = None
    try:
        names = ds.features['category'].names
    except Exception:
        try:
            names = ds.features['category'].feature.names
        except Exception:
            pass
    assert names, 'no category names in dataset features — inspect ds.features and adjust'

    per_obj = []
    for ex in ds:
        cats = ex.get('category')
        boxes = ex.get('bbox')
        img = ex['image']
        if cats is None or boxes is None:
            continue
        if isinstance(cats, int):
            cats = [cats]
        if not isinstance(boxes[0], (list, tuple)):
            boxes = [boxes]
        for ci, bb in zip(cats, boxes):
            lab = names[ci] if isinstance(ci, int) else str(ci)
            slot = FP_LABELS_TO_SLOT.get(lab)
            if slot:
                per_obj.append((img, bb, slot))
    print('mapped objects:', len(per_obj), '| slots:', Counter(s for _, _, s in per_obj))

    N_PER_SLOT = 1500
    by_slot = defaultdict(list)
    for item in per_obj:
        by_slot[item[2]].append(item)
    picked, y_true = [], []
    for s, items in by_slot.items():
        idx = rng.permutation(len(items))[:N_PER_SLOT]
        for i in idx:
            picked.append(items[i]); y_true.append(s)

    from PIL import Image as _PILImage
    pils = []
    for img, bb, s in picked:
        try:
            im = img if isinstance(img, _PILImage.Image) else _PILImage.open(io.BytesIO(img['bytes'])).convert('RGB')
            x1, y1, x2, y2 = [float(v) for v in bb]
            W, H = im.size
            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(W, int(x2)), min(H, int(y2))
            if x2 - x1 < 8 or y2 - y1 < 8:
                continue
            pils.append(im.crop((x1, y1, x2, y2)).resize((256, 256)))
        except Exception:
            continue

    if pils:
        emb = embed_images(pils)
        labels = y_true[:len(pils)]
        RESULTS[NAME] = {'zero-shot': score(labels, pred_zero_shot(emb)),
                         'probe': score(labels, pred_probe(emb))}
        print('probe acc:', RESULTS[NAME]['probe']['acc'])
        print(json.dumps(RESULTS[NAME]['probe']['per_slot'], indent=1))
    else:
        print('no Fashionpedia objects mapped — check label names and adjust FP_LABELS_TO_SLOT')
except Exception as e:
    print('Fashionpedia skipped:', type(e).__name__, str(e)[:300])


## 4. ModaNet — street realism (experimental)

Annotations are public on GitHub (`eBay/modanet`, git-lfs); the images come from
Chictopia/PaperDoll and have **no bulk public download**. This cell fetches annotations,
maps the 13 meta-categories, and tries a small image pull; expect it to degrade to a skip —
if so, the annotations still serve T-009 background-augmentation labels.


In [ ]:
NAME = 'modanet'
MN_LABELS = ['bag', 'belt', 'boots', 'footwear', 'outer', 'dress', 'sunglasses',
             'pants', 'top', 'shorts', 'skirt', 'headwear', 'scarf/tie']
MN2SLOT = {
    'top': 'top', 'pants': 'bottom', 'shorts': 'bottom', 'skirt': 'bottom',
    'dress': 'dress', 'outer': 'outerwear',
    'boots': 'shoes', 'footwear': 'shoes',
    'bag': 'accessory', 'belt': 'accessory', 'sunglasses': 'accessory',
    'headwear': 'accessory', 'scarf/tie': 'accessory',
}
try:
    work = os.path.join(RAW, 'modanet')
    ann = None
    for cand in glob.glob(os.path.join(work, '**', '*.json*'), recursive=True):
        ann = cand; break
    if ann is None:
        os.makedirs(work, exist_ok=True)
        sh(['git', 'clone', '--depth', '1', 'https://github.com/eBay/modanet.git', work])
        found = glob.glob(os.path.join(work, '**', '*.json*'), recursive=True)
        print('annotation files:', [os.path.basename(f) for f in found])
        ann = found[0] if found else None
    assert ann, 'no ModaNet annotations available'
    print('annotations:', ann)

    img_urls = {}   # image_id -> url
    ann_data = json.load(open(ann)) if ann.endswith('.json') else [json.loads(l) for l in open(ann)]
    for item in ann_data if isinstance(ann_data, list) else ann_data.get('images', []):
        iid = item.get('id'); u = item.get('coco_url') or item.get('file_name')
        if iid and u:
            img_urls[iid] = u
    print('annotated images:', len(img_urls))

    # Image bulk download is not publicly scriptable (PaperDoll/Chictopia are gone).
    # Score only if a local image mirror already exists (e.g. user-provided zip in raw/modanet).
    mirror = os.path.join(work, 'images')
    if os.path.isdir(mirror) and len(os.listdir(mirror)) > 100:
        raise RuntimeError('local mirror found but per-object mapping not wired yet — TODO with real data')
    print('ModaNet: annotations fetched (usable as T-009 augmentation labels); '
          'image eval SKIPPED — no public bulk image source. See README notes.')
except Exception as e:
    print('ModaNet skipped:', type(e).__name__, str(e)[:300])


In [ ]:
# ---- Combined report ----
def fmt_table():
    lines = ['# StyleMe — dataset slot benchmark (notebook 04)', '',
             '| dataset | model | acc | n | per-slot recall |', '|---|---|---|---|---|']
    for ds, models in RESULTS.items():
        for m, s in models.items():
            rec = ', '.join(f"{k} {v['recall']:.2f}" for k, v in s['per_slot'].items())
            lines.append(f"| {ds} | {m} | {s['acc']:.3f} | {s['n']} | {rec} |")
    skipped = [d for d in ('deepfashion2', 'fashion_product', 'fashionpedia', 'modanet') if d not in RESULTS]
    if skipped:
        lines += ['', 'Skipped: ' + ', '.join(skipped)]
    lines += ['', 'Gate: probe >= 0.90 on the DeepFashion2 anchor (matches ml/README results table).',
              'Street sets (Fashionpedia/ModaNet) measure distribution shift, not the gate.',
              f'Sampling seed pinned at {SEED} — numbers stay comparable across T-009 retrains.',
              'All corpora non-commercial licensed — eval/research use only.']
    return chr(10).join(lines)

report_md = fmt_table()
print(report_md)
json.dump({'seed': SEED,
           'results': RESULTS,
           'skipped': [d for d in ('deepfashion2', 'fashion_product', 'fashionpedia', 'modanet')
                       if d not in RESULTS]},
          open(os.path.join(ART, 'dataset_eval_report.json'), 'w'), indent=1)
open(os.path.join(ART, 'dataset_eval_report.md'), 'w').write(report_md)
print('wrote artifacts/dataset_eval_report.{json,md}')


## Next

- Numbers land in `artifacts/dataset_eval_report.md` after a Colab run (mirrors notebook 01's gate table).
- Fashionpedia/ModaNet per-slot recall on street photos decides how far on-device "worn" captures
  (Q-003) can lean on the shipped probe vs needing T-009's background-augmented retrain.
- ModaNet annotations double as T-009 augmentation labels even without image eval.
